In [6]:
import pandas as pd
import numpy as np
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots

import nbformat 
#import kaleido

In [8]:
# cargo CSV precio del bitcoin
btc = pd.read_csv("../data/btc.csv")
# elimino fila 0, donde aparece btc-usd
btc = btc[btc["Close"] != "BTC-USD"]

# convierto Date a datetime
btc['Date'] = pd.to_datetime(btc['Date'], errors='coerce')

# convierto las demás columnas a numéricas (float)
numeric_cols = ['Close', 'High', 'Low', 'Open', 'Volume']
for col in numeric_cols:
    btc[col] = pd.to_numeric(btc[col], errors='coerce')

# reviso cómo quedaron
btc.info()
btc.isnull().sum()

# primera descripcion
btc.describe()
btc.tail()

<class 'pandas.core.frame.DataFrame'>
Index: 3113 entries, 1 to 3113
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    3113 non-null   datetime64[ns]
 1   Close   3113 non-null   float64       
 2   High    3113 non-null   float64       
 3   Low     3113 non-null   float64       
 4   Open    3113 non-null   float64       
 5   Volume  3113 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 170.2 KB


,Date,Close,High,Low,Open,Volume
3109,2026-08-06,64262.113281,64934.492188,64098.476562,64595.449219,18529402711
3110,2026-08-07,64880.191406,65330.609375,64113.273438,64257.488281,22165720102
3111,2026-08-08,64904.687500,65140.480469,64797.113281,64882.546875,12350094271
3112,2026-08-09,64844.886719,65401.691406,64677.601562,64906.550781,13234538380
3113,2026-08-10,64043.179688,65278.343750,63764.757812,64848.906250,23706193920


In [9]:
# Agrego columnas calculadas
btc['Daily_Change'] = btc['Close'] - btc['Open'] #Diferencia entre el precio de cierre y de apertura del día.
btc['Volatility'] = btc['High'] - btc['Low'] #Diferencia entre el valor máximo y mínimo del día.
btc['Pct_Change'] = btc['Close'].pct_change() #variación porcentual del precio de cierre respecto al día anterior.
btc['Volume_Change_pct'] = btc["Volume"].pct_change() #Variación porcentual del volumen de transacciones respecto al día anterior.
btc['SMA_7'] = btc["Close"].rolling(7).mean() #Promedio/media móvil a 7 días. Tendencia a corto plazo.
btc['SMA_30'] = btc["Close"].rolling(30).mean() #Promedio/media móvil a 30 días. Tendencia a largo plazo.
btc["Rolling_volatility_30"] = btc["Pct_Change"].rolling(30).std() #desviacion los ultimos 30 dias
lags = [1, 2, 3, 7]  # días anteriores

# Crear las columnas de lags
for lag in lags:
    btc[f'BTC_Close_t-{lag}'] = btc['Close'].shift(lag)


btc.head()
btc.tail(10)

,Date,Close,High,Low,Open,Volume,Daily_Change,Volatility,Pct_Change,Volume_Change_pct,SMA_7,SMA_30,Rolling_volatility_30,BTC_Close_t-1,BTC_Close_t-2,BTC_Close_t-3,BTC_Close_t-7
3104,2026-08-01,62763.320312,63091.597656,62233.011719,62813.664062,13553286543,-50.343750,858.585938,-0.000803,-0.554750,63878.157924,64037.761328,0.015629,62813.746094,64725.308594,63908.167969,64311.812500
3105,2026-08-02,63482.000000,63714.613281,62745.253906,62763.445312,15901776708,718.554688,969.359375,0.011451,0.173278,63612.686384,64069.021354,0.015455,62763.320312,62813.746094,64725.308594,65340.300781
3106,2026-08-03,63460.898438,64020.324219,62226.578125,63485.441406,26038889824,-24.542969,1793.746094,-0.000332,0.637483,63574.972098,64081.441146,0.015379,63482.000000,62763.320312,62813.746094,63724.898438
3107,2026-08-04,64055.953125,64466.937500,63277.683594,63458.410156,23662357784,597.542969,1189.253906,0.009377,-0.091269,63601.342076,64098.376823,0.015417,63460.898438,63482.000000,62763.320312,63871.363281
3108,2026-08-05,64597.500000,64954.347656,63829.718750,64054.878906,23568584891,542.621094,1124.628906,0.008454,-0.003963,63699.818080,64118.459635,0.015440,64055.953125,63460.898438,63482.000000,63908.167969
3109,2026-08-06,64262.113281,64934.492188,64098.476562,64595.449219,18529402711,-333.335938,836.015625,-0.005192,-0.213809,63633.647321,64150.616927,0.015330,64597.500000,64055.953125,63460.898438,64725.308594
3110,2026-08-07,64880.191406,65330.609375,64113.273438,64257.488281,22165720102,622.703125,1217.335938,0.009618,0.196246,63928.853795,64238.037500,0.015067,64262.113281,64597.500000,64055.953125,62813.746094
3111,2026-08-08,64904.687500,65140.480469,64797.113281,64882.546875,12350094271,22.140625,343.367188,0.000378,-0.442829,64234.763393,64295.088802,0.014849,64880.191406,64262.113281,64597.500000,62763.320312
3112,2026-08-09,64844.886719,65401.691406,64677.601562,64906.550781,13234538380,-61.664062,724.089844,-0.000921,0.071614,64429.461496,64319.013672,0.014621,64904.687500,64880.191406,64262.113281,63482.000000
3113,2026-08-10,64043.179688,65278.343750,63764.757812,64848.906250,23706193920,-805.726562,1513.585938,-0.012363,0.791237,64512.644531,64327.052083,0.014776,64844.886719,64904.687500,64880.191406,63460.898438


In [10]:
# Cargar el índice Fear and Greed
fear_greed = pd.read_csv("../data/fear_greed.csv")

# fear_greed = fear_greed.drop(columns=['time_until_update'])
fear_greed = fear_greed.rename(columns={'date': 'Date'})

# Convertir a datetime desde formato día-mes-año
fear_greed['Date'] = pd.to_datetime(
    fear_greed['Date'].astype(str).str.strip(),
    format='%d-%m-%Y',
    errors='coerce'
)

fear_greed['Date'] = fear_greed['Date'].dt.tz_localize(None)
fear_greed['Date'] = fear_greed['Date'].dt.normalize()
fear_greed = fear_greed.sort_values(by='Date', ascending=True)
fear_greed = fear_greed.reset_index(drop=True)

# Mostrar información y primeros registros
fear_greed.describe()
fear_greed.info()
fear_greed.tail()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3109 entries, 0 to 3108
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                3109 non-null   datetime64[ns]
 1   fng_value           3109 non-null   float64       
 2   fng_classification  3109 non-null   object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 73.0+ KB


,Date,fng_value,fng_classification
3104,2026-08-06,25.0,Extreme Fear
3105,2026-08-07,29.0,Fear
3106,2026-08-08,30.0,Fear
3107,2026-08-09,31.0,Fear
3108,2026-08-10,30.0,Fear


In [11]:
# Agrego columnas calculadas:
fear_greed['fng_diff_day'] = fear_greed['fng_value'].diff()
fear_greed['fng_SMA_7'] = fear_greed['fng_value'].rolling(7).mean()
fear_greed['fng_SMA_30'] = fear_greed['fng_value'].rolling(30).mean()
fear_greed['fng_trend'] = fear_greed['fng_SMA_7'] - fear_greed['fng_SMA_30']
fear_greed.head(100)

,Date,fng_value,fng_classification,fng_diff_day,fng_SMA_7,fng_SMA_30,fng_trend
0,2018-02-01,30.0,Fear,NaN,NaN,NaN,NaN
1,2018-02-02,15.0,Extreme Fear,-15.0,NaN,NaN,NaN
2,2018-02-03,40.0,Fear,25.0,NaN,NaN,NaN
3,2018-02-04,24.0,Extreme Fear,-16.0,NaN,NaN,NaN
4,2018-02-05,11.0,Extreme Fear,-13.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...
95,2018-05-10,63.0,Greed,10.0,60.000000,42.033333,17.966667
96,2018-05-11,41.0,Fear,-22.0,57.857143,42.833333,15.023810
97,2018-05-12,44.0,Fear,3.0,55.142857,43.600000,11.542857
98,2018-05-13,40.0,Fear,-4.0,51.285714,44.333333,6.952381


In [13]:
full_data = pd.merge_asof(btc.sort_values('Date'),
                          fear_greed.sort_values('Date'),
                          on = 'Date',
                          direction='backward')
full_data.head(10)
full_data.tail(10)

,Date,Close,High,Low,Open,Volume,Daily_Change,Volatility,Pct_Change,Volume_Change_pct,...,BTC_Close_t-1,BTC_Close_t-2,BTC_Close_t-3,BTC_Close_t-7,fng_value,fng_classification,fng_diff_day,fng_SMA_7,fng_SMA_30,fng_trend
3103,2026-08-01,62763.320312,63091.597656,62233.011719,62813.664062,13553286543,-50.343750,858.585938,-0.000803,-0.554750,...,62813.746094,64725.308594,63908.167969,64311.812500,27.0,Fear,2.0,27.714286,26.033333,1.680952
3104,2026-08-02,63482.000000,63714.613281,62745.253906,62763.445312,15901776708,718.554688,969.359375,0.011451,0.173278,...,62763.320312,62813.746094,64725.308594,65340.300781,27.0,Fear,0.0,27.857143,26.233333,1.623810
3105,2026-08-03,63460.898438,64020.324219,62226.578125,63485.441406,26038889824,-24.542969,1793.746094,-0.000332,0.637483,...,63482.000000,62763.320312,62813.746094,63724.898438,28.0,Fear,1.0,27.571429,26.433333,1.138095
3106,2026-08-04,64055.953125,64466.937500,63277.683594,63458.410156,23662357784,597.542969,1189.253906,0.009377,-0.091269,...,63460.898438,63482.000000,62763.320312,63871.363281,25.0,Extreme Fear,-3.0,27.000000,26.500000,0.500000
3107,2026-08-05,64597.500000,64954.347656,63829.718750,64054.878906,23568584891,542.621094,1124.628906,0.008454,-0.003963,...,64055.953125,63460.898438,63482.000000,63908.167969,27.0,Fear,2.0,26.714286,26.600000,0.114286
3108,2026-08-06,64262.113281,64934.492188,64098.476562,64595.449219,18529402711,-333.335938,836.015625,-0.005192,-0.213809,...,64597.500000,64055.953125,63460.898438,64725.308594,25.0,Extreme Fear,-2.0,26.285714,26.533333,-0.247619
3109,2026-08-07,64880.191406,65330.609375,64113.273438,64257.488281,22165720102,622.703125,1217.335938,0.009618,0.196246,...,64262.113281,64597.500000,64055.953125,62813.746094,29.0,Fear,4.0,26.857143,26.833333,0.023810
3110,2026-08-08,64904.687500,65140.480469,64797.113281,64882.546875,12350094271,22.140625,343.367188,0.000378,-0.442829,...,64880.191406,64262.113281,64597.500000,62763.320312,30.0,Fear,1.0,27.285714,27.100000,0.185714
3111,2026-08-09,64844.886719,65401.691406,64677.601562,64906.550781,13234538380,-61.664062,724.089844,-0.000921,0.071614,...,64904.687500,64880.191406,64262.113281,63482.000000,31.0,Fear,1.0,27.857143,27.366667,0.490476
3112,2026-08-10,64043.179688,65278.343750,63764.757812,64848.906250,23706193920,-805.726562,1513.585938,-0.012363,0.791237,...,64844.886719,64904.687500,64880.191406,63460.898438,30.0,Fear,-1.0,28.142857,27.500000,0.642857


# Halving de Bitcoin

El halving es un evento programado que reduce a la mitad la recompensa por minar bloques, aproximadamente cada 4 años.

## Impacto en BTC

- **Menor oferta nueva:** ingresan menos bitcoins al mercado, lo que aumenta la escasez.
- **Efecto en el precio:** si la demanda se mantiene o crece, esa menor oferta puede impulsar subas en el precio.
- **Expectativa del mercado:** suele generar anticipación y mayor atención de inversores antes y después del evento.
- **Impacto en mineros:** reduce ingresos por bloque y puede afectar la rentabilidad de los mineros menos eficientes.

En ciclos anteriores, los halvings estuvieron seguidos por etapas de fuerte apreciación del precio, aunque no garantizan resultados futuros.

In [14]:
halvings = [pd.Timestamp("2012-11-28"),
            pd.Timestamp("2016-07-09"),
            pd.Timestamp("2020-05-11"),
            pd.Timestamp("2024-04-20")
           ]

rewards = [25, 12.5, 6.25, 3.125]

full_data["Is_Halving_Date"] = 0
full_data["Block_reward"] = np.nan

full_data.loc[full_data["Date"].isin(halvings), "Is_Halving_Date"] = 1

for i in range(len(halvings)):
    start = halvings[i]
    end = halvings[i+1] if i+1 < len(halvings) else full_data["Date"].max()

    mask = (full_data["Date"] >= start) & (full_data["Date"] < end)

    full_data.loc[mask, "Block_reward"] = rewards[i]


full_data.sample(10)

,Date,Close,High,Low,Open,Volume,Daily_Change,Volatility,Pct_Change,Volume_Change_pct,...,BTC_Close_t-3,BTC_Close_t-7,fng_value,fng_classification,fng_diff_day,fng_SMA_7,fng_SMA_30,fng_trend,Is_Halving_Date,Block_reward
50,2018-03-23,8879.620117,8879.620117,8360.620117,8736.250000,5954120192,143.370117,519.000000,0.017317,0.076618,...,8913.469727,8338.349609,28.0,Fear,-8.0,32.285714,39.100000,-6.814286,0,12.500
2385,2024-08-13,60609.566406,61572.398438,58506.253906,59356.207031,30327698167,1253.359375,3066.144531,0.021145,-0.182071,...,60945.812500,56034.316406,31.0,Fear,6.0,33.142857,53.366667,-20.223810,0,3.125
2545,2025-01-20,102016.664062,109114.882812,99471.359375,101083.750000,126279678351,932.914062,9643.523438,0.009171,0.644482,...,104462.039062,94516.523438,76.0,Extreme Greed,-1.0,73.285714,70.633333,2.652381,0,3.125
187,2018-08-07,6753.120117,7146.560059,6748.240234,6958.319824,4682800000,-205.199707,398.319824,-0.028580,0.192797,...,7032.850098,7780.439941,25.0,Extreme Fear,0.0,31.142857,39.966667,-8.823810,0,12.500
2542,2025-01-17,104462.039062,105884.226562,99948.906250,100025.765625,71888972663,4436.273438,5935.320312,0.047166,0.328724,...,96534.046875,94701.453125,75.0,Greed,0.0,67.857143,70.366667,-2.509524,0,3.125
1375,2021-11-07,63326.988281,63326.988281,61432.488281,61554.921875,24726754302,1772.066406,1894.500000,0.029247,-0.150135,...,61452.230469,61318.957031,73.0,Greed,2.0,73.285714,73.933333,-0.647619,0,6.250
586,2019-09-10,10115.975586,10394.353516,10020.573242,10336.408203,14906809639,-220.432617,373.780273,-0.021190,-0.152827,...,10517.254883,10623.540039,41.0,Fear,0.0,41.571429,30.766667,10.804762,0,12.500
1815,2023-01-21,22777.625000,23282.347656,22511.833984,22677.427734,32442278429,100.197266,770.513672,0.004457,0.126501,...,20688.781250,20976.298828,53.0,Neutral,2.0,49.857143,33.166667,16.690476,0,6.250
1814,2023-01-20,22676.552734,22692.357422,20919.126953,21085.373047,28799154319,1591.179688,1773.230469,0.075391,0.361479,...,21161.519531,19909.574219,51.0,Neutral,6.0,48.857143,32.333333,16.523810,0,6.250
403,2019-03-11,3905.227295,3966.384766,3889.239014,3953.740234,10125901903,-48.512939,77.145752,-0.011735,0.042482,...,3901.131592,3761.557129,56.0,Greed,1.0,50.428571,48.800000,1.628571,0,12.500


In [15]:
#Guardo el csv final para entrenal 
full_data.to_csv("../Data/full_data.csv",index=False)
full_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3113 entries, 0 to 3112
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Date                   3113 non-null   datetime64[ns]
 1   Close                  3113 non-null   float64       
 2   High                   3113 non-null   float64       
 3   Low                    3113 non-null   float64       
 4   Open                   3113 non-null   float64       
 5   Volume                 3113 non-null   int64         
 6   Daily_Change           3113 non-null   float64       
 7   Volatility             3113 non-null   float64       
 8   Pct_Change             3112 non-null   float64       
 9   Volume_Change_pct      3112 non-null   float64       
 10  SMA_7                  3107 non-null   float64       
 11  SMA_30                 3084 non-null   float64       
 12  Rolling_volatility_30  3083 non-null   float64       
 13  BTC

In [16]:
full_data["Date"] = pd.to_datetime(full_data["Date"])  # asegurar tipo datetime
full_data= full_data.set_index("Date", drop=False)

In [18]:
# Seleciono las columnas del indice F&G para hacer el mapa de calor (correlacion)
fng_cols = ['fng_value', 'fng_diff_day', 'fng_SMA_7', 'fng_SMA_30', 'fng_trend']

# Columnas de precio BTC relevantes
btc_cols = ['Close', 'Daily_Change', 'Volatility', 'Pct_Change', 'Volume_Change_pct', 'SMA_7', 'SMA_30', 'Rolling_volatility_30', 'Block_reward']

# Selecciono solo las columnas de interés
corr_data = full_data[fng_cols + btc_cols].dropna()

# Calculo la correlación
corr_matrix = corr_data.corr()

# Heatmap interactivo con plotly
fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Mapa de Calor - Correlación Fear & Greed vs BTC'
)
fig.show()

In [19]:
# Crear figura
fig = go.Figure()

# Precio del Bitcoin
fig.add_trace(go.Scatter(
    x=full_data["Date"], 
    y=full_data["Close"],
    mode='lines', 
    name='BTC Price', 
    line=dict(color='blue')
))

# Block reward (como escalón)
fig.add_trace(go.Scatter(
    x=full_data["Date"], 
    y=full_data["Block_reward"],
    mode='lines', 
    name='Block Reward', 
    line=dict(color='red', dash='dash'),
    yaxis="y2"
))

# Puntos de halving
halving_points = full_data[full_data["Is_Halving_Date"] == 1]

fig.add_trace(go.Scatter(
    x=halving_points["Date"],
    y=halving_points["Close"],
    mode="markers+text",
    name="Halving",
    marker=dict(size=10, color="green", symbol="diamond"),
    text=halving_points["Close"].round(0),   # precio aproximado como etiqueta
    textposition="top center"
))

# Layout
fig.update_layout(
    title="Bitcoin Price vs Block Reward con Halvings",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    yaxis2=dict(title="Block Reward (BTC)", overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99)
)

fig.show()

## Fear and Greed (índice de miedo y codicia)  
Es una medida del sentimiento del mercado, creada para el mercado de acciones, pero que también se aplica a criptomonedas. Su objetivo es reflejar si los inversores están dominados por el miedo o por la codicia, lo que puede influir en sus decisiones de compra o venta.  
El índice va de 0 a 100:  
- 0-25: Miedo extremo
- 26-49: Miedo
- 50-74: Codicia
- 75-100: Codicia extrema  

El gráfico refleja este patrón, puntos azules (Fear/Extreme Fear) coinciden con caídas del precio del bitcoin, y puntos rojos (Greed/Extreme Greed) coincide con picos de subida en el precio.

In [20]:
# Mapeo de colores para las categorías del Fear & Greed
colors = {
    'Extreme Fear': 'darkblue',
    'Fear': 'blue',
    'Neutral': 'orange',
    'Greed': 'red',
    'Extreme Greed': 'darkred'
}

df_plot = full_data[['Date', 'Close', 'fng_value', 'fng_classification']].dropna()

fig = go.Figure()

# Línea de precio de Bitcoin
fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Close'],
    mode='lines',
    name='Precio BTC',
    line=dict(color='black', width=2)
))

# Marcadores del Fear & Greed
fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['fng_value'],
    mode='markers',
    name='Fear & Greed',
    marker=dict(color=[colors[c] for c in df_plot['fng_classification']], size=10),
    text=df_plot['fng_classification'],  # Muestra la categoría al pasar el cursor
    hoverinfo='text+y'
))

# Layout
fig.update_layout(
    title='Precio de Bitcoin vs Fear & Greed',
    yaxis=dict(title='Precio BTC', side='left'),
    yaxis2=dict(title='Fear & Greed Value', overlaying='y', side='right', range=[0,100]),
    legend=dict(x=0.01, y=0.99)
)

fig.show()

**Visualización de tendencia en este último año**  
*SMA 7 encima de SMA 30 → tendencia alcista: el precio reciente está subiendo más rápido que la tendencia general.*   
*SMA 7 debajo de SMA 30 → tendencia bajista: el precio reciente está cayendo respecto a la tendencia general.*
    
Entre enero y marzo de 2026, el precio muestra una tendencia bajista marcada, cayendo desde un pico inicial superior a los 90 000 USD (tras el máximo del ciclo a fines del año pasado) hasta tocar un soporte crítico alrededor de los 60 000 USD.
Desde abril hasta mediados de mayo, se observa una fuerte recuperación temporal, con precios que logran repuntar con fuerza superando los 75 000 USD y buscando estabilizarse. 
En junio, el precio experimenta nuevos altibajos y una corrección que vuelve a testear la zona de soporte clave de los 60 000–65 000 USD, manteniendo al mercado a la expectativa.

*Cruce hacia arriba (SMA 7 cruza SMA 30 desde abajo): señal alcista y de compra potencial (“golden cross”).*  
*Cruce hacia abajo (SMA 7 cruza SMA 30 desde arriba): señal bajista y de venta potencial (“death cross”).*


**Tendencia actual (junio 2026):** El precio consolida cerca del soporte de los 66 000 USD, mostrando divergencias alcistas en temporalidades largas. La media de 7 días intenta estabilizarse sobre la de 30 días, lo que sugiere signos de un posible cambio de tendencia si logra romper las resistencias inmediatas.

In [22]:
fig = go.Figure()

df_short = full_data[full_data['Date'] >= '2026-01-01']

# Precio de cierre BTC
fig.add_trace(go.Scatter(
    x=df_short['Date'],
    y=df_short['Close'],
    mode='lines',
    name='BTC Close',
    line=dict(color='black', width=2)
))

# SMA 7 días
fig.add_trace(go.Scatter(
    x=df_short['Date'],
    y=df_short['SMA_7'],
    mode='lines',
    name='SMA 7 días',
    line=dict(color='blue', width=2, dash='dash')
))

# SMA 30 días
fig.add_trace(go.Scatter(
    x=df_short['Date'],
    y=df_short['SMA_30'],
    mode='lines',
    name='SMA 30 días',
    line=dict(color='red', width=2, dash='dash')
))

# Layout
fig.update_layout(
    title="Precio de Bitcoin con Promedios Móviles (7 y 30 días) desde ene-2025 a hoy",
    xaxis_title="Fecha",
    yaxis_title="Precio USD",
    template="plotly_white",
    width=900,
    height=500,
    hovermode="x unified"
)

fig.show()

In [23]:
full_data.describe()


,Date,Close,High,Low,Open,Volume,Daily_Change,Volatility,Pct_Change,Volume_Change_pct,...,BTC_Close_t-2,BTC_Close_t-3,BTC_Close_t-7,fng_value,fng_diff_day,fng_SMA_7,fng_SMA_30,fng_trend,Is_Halving_Date,Block_reward
count,3113,3113.000000,3113.000000,3113.000000,3113.000000,3.113000e+03,3113.000000,3113.000000,3112.000000,3112.000000,...,3111.000000,3110.000000,3106.000000,3113.000000,3112.000000,3107.000000,3084.000000,3084.000000,3113.000000,3112.000000
mean,2022-05-06 23:59:59.999999744,40004.664127,40756.829102,39168.368654,39988.649154,3.078628e+10,16.014973,1588.460448,0.001169,0.053437,...,39988.952543,39980.941053,39949.430430,45.380983,0.005784,45.412847,45.490510,-0.076669,0.000642,7.071417
min,2018-02-01 00:00:00,3236.761719,3275.377930,3191.303467,3236.274658,2.923670e+09,-10314.277344,18.922607,-0.371695,-0.869188,...,3236.761719,3236.761719,3236.761719,5.000000,-45.000000,7.428571,10.000000,-31.338095,0.000000,3.125000
25%,2020-03-20 00:00:00,9733.721680,9938.297852,9546.969727,9729.321289,1.603826e+10,-343.882812,358.294922,-0.013417,-0.141645,...,9731.761719,9730.781738,9729.443604,26.000000,-3.000000,26.714286,28.366667,-5.130952,0.000000,3.125000
50%,2022-05-07 00:00:00,30111.998047,30555.537109,29527.740234,30098.585938,2.689636e+10,6.523438,1050.310547,0.000618,-0.007800,...,30101.265625,30093.755859,30055.917969,44.000000,0.000000,43.714286,43.683333,0.066667,0.000000,6.250000
75%,2024-06-23 00:00:00,63239.519531,64321.484375,62117.410156,63244.085938,4.035348e+10,387.799805,2331.843750,0.014909,0.175461,...,63231.970703,63218.088867,63189.931641,64.000000,3.000000,63.142857,61.741667,5.394048,0.000000,12.500000
max,2026-08-10 00:00:00,124752.531250,126198.070312,123196.046875,124752.140625,3.509679e+11,8230.070312,17927.250000,0.187465,5.439003,...,124752.531250,124752.531250,124752.531250,95.000000,40.000000,94.000000,92.366667,27.123810,1.000000,12.500000
std,NaN,32463.053458,32978.899275,31906.236777,32465.071652,2.170447e+10,1340.373680,1651.391851,0.032803,0.375342,...,32467.570002,32469.715475,32478.734973,22.063199,6.432647,21.305071,19.859276,9.203099,0.025343,3.519920
